# **Importing GTSAM**

In [ ]:
# Install gtsam-develop if not installed
try:
    import gtsam
except ImportError:
    %pip install gtsam-develop

In [ ]:
# Imports
import gtsam
import gtsam.utils.plot as gtsam_plot
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from gtsam.symbol_shorthand import X, V, B

# **IMU Factor Simple Test**

A simple test with no motion on an IMU with a IMUFactor

In [ ]:
# Define IMU parameters (example values)
gravity = np.array([0, 0, -9.81])
params = gtsam.PreintegrationParams(gravity)
params.setAccelerometerCovariance(np.eye(3) * 1e-4)
params.setGyroscopeCovariance(np.eye(3) * 1e-4)
params.setIntegrationCovariance(np.eye(3) * 1e-5)

In [ ]:
# Simulate IMU measurements
# Let's assume a simple motion for simulation purposes
dt = 0.1
measured_acc = np.array([0, 0, 0])  # No acceleration
measured_gyro = np.array([0, 0, 0]) # No rotation

In [ ]:
# Initial state (pose, velocity, bias) at t=0
pose_0 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(0, 0, 0))
vel_0 = np.array([0, 0, 0.0])
bias_0 = gtsam.imuBias.ConstantBias(np.array([0, 0, 0.0]), np.array([0, 0, 0.0]))

In [ ]:
# Create a factor graph
graph = gtsam.NonlinearFactorGraph()

# Define noise models
pose_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.3, 0.3, 0.3]))  # Pose noise (x, y, z, roll, pitch, yaw)
velocity_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1]))
bias_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.01, 0.01, 0.01, 0.01, 0.01, 0.01])) # Bias noise (accel, gyro)

# Add priors
graph.add(gtsam.PriorFactorPose3(X(0), pose_0, pose_noise))
graph.add(gtsam.PriorFactorVector(V(0), vel_0, velocity_noise))
graph.add(gtsam.PriorFactorConstantBias(B(0), bias_0, bias_noise))

# Preintegrate IMU measurements for interval 1
# With bias correction as we are assuming bias_0 is not zero
pim = gtsam.PreintegratedImuMeasurements(params, bias_0)
for i in range(10):
  pim.integrateMeasurement(measured_acc, measured_gyro, dt)

# Add ImuFactor for interval 1 (t=0 to t=1)
graph.add(gtsam.ImuFactor(X(0), V(0),
                          X(1), V(1),
                          B(1), pim))

# Add the bias random walk model between B(0) and B(1)
bias_random_walk_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.01, 0.01, 0.01, 0.01, 0.01, 0.01])) # Bias random walk noise
graph.add(gtsam.BetweenFactorConstantBias(B(0), B(1), gtsam.imuBias.ConstantBias(), bias_random_walk_noise))



In [ ]:
# Add ImuFactor for interval 2 (t=1 to t=2)
graph.add(gtsam.ImuFactor(X(1), V(1),
                          X(2), V(2),
                          B(2), pim))
# Add the bias random walk model between B(1) and B(2)
graph.add(gtsam.BetweenFactorConstantBias(B(1), B(2), gtsam.imuBias.ConstantBias(), bias_random_walk_noise))

In [ ]:
# Create initial estimate
initial_estimate = gtsam.Values()
initial_estimate.insert(X(0), pose_0)
initial_estimate.insert(V(0), vel_0)

# Initial estimate for t=1, t=2 (assuming constant velocity)
for k in [1,2]:
  pose_k = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(0, 0, 0) + vel_0 * dt)
  vel_k = vel_0
  initial_estimate.insert(X(k), pose_k)
  initial_estimate.insert(V(k), vel_k)

# Initial estimate for bias in interval 1 and 2:
initial_estimate.insert(B(0), bias_0)
initial_estimate.insert(B(1), bias_0)
initial_estimate.insert(B(2), bias_0)
print("Initial Estimate:\n", initial_estimate)

In [ ]:
# Optimize the graph
optimizer = gtsam.GaussNewtonOptimizer(graph, initial_estimate) # Force error if indeterminate
result = optimizer.optimize()

# Print results
print("\nOptimized Result:\n", result)

In [ ]:
# Analyze marginals (covariances)
marginals = gtsam.Marginals(graph, result)

# Get covariance for bias in interval 1
bias_cov_1 = marginals.marginalCovariance(gtsam.symbol_shorthand.B(1))
print("\nCovariance for Bias at t=1:\n", np.round(1e9*bias_cov_1))

# Get covariance for bias in interval 2
bias_cov_2 = marginals.marginalCovariance(gtsam.symbol_shorthand.B(2))
print("\nCovariance for Bias at t=2:\n", np.round(1e9*bias_cov_2))

In [ ]:
# The entire posterior over *all* variables is encoded in the Bayes Tree,
# at the end of this print. From it all marginals are computed.
marginals.print()

# **Combined IMU Factor Simple Test**

In [ ]:
# Define IMU parameters (example values)
gravity = np.array([0, 0, -9.81])
params = gtsam.PreintegrationCombinedParams(gravity)
params.setAccelerometerCovariance(np.eye(3) * 1e-4)
params.setGyroscopeCovariance(np.eye(3) * 1e-4)
params.setIntegrationCovariance(np.eye(3) * 1e-5)
params.setBiasAccCovariance(np.eye(3) * 0.0001)
params.setBiasOmegaCovariance(np.eye(3) * 0.0001)
params.setBiasAccOmegaInit(np.eye(6) * 1e-9) #small number to prevent numerical singularity issues

In [ ]:
# Simulate IMU measurements
# Let's assume a simple motion for simulation purposes
dt = 0.1
measured_acc = np.array([0, 0, 0])  # No acceleration
measured_gyro = np.array([0, 0, 0]) # No rotation

In [ ]:
# Initial state (pose, velocity, bias) at t=0
pose_0 = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(0, 0, 0))
vel_0 = np.array([0, 0, 0.0])
bias_0 = gtsam.imuBias.ConstantBias(np.array([0, 0, 0.0]), np.array([0, 0, 0.0]))

In [ ]:
# Create a factor graph
graph = gtsam.NonlinearFactorGraph()

# Define noise models
pose_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.3, 0.3, 0.3]))  # Pose noise (x, y, z, roll, pitch, yaw)
velocity_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1]))
bias_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.01, 0.01, 0.01, 0.01, 0.01, 0.01])) # Bias noise (accel, gyro)

# Add priors
graph.add(gtsam.PriorFactorPose3(X(0), pose_0, pose_noise))
graph.add(gtsam.PriorFactorVector(V(0), vel_0, velocity_noise))
graph.add(gtsam.PriorFactorConstantBias(B(0), bias_0, bias_noise))

# Preintegrate IMU measurements for interval 1
# With bias correction as we are assuming bias_0 is not zero
pim = gtsam.PreintegratedCombinedMeasurements(params, bias_0)
for i in range(10):
  pim.integrateMeasurement(measured_acc, measured_gyro, dt)

# Add ImuFactor for interval 1 (t=0 to t=1)
graph.add(gtsam.CombinedImuFactor(X(0), V(0),
                          X(1), V(1),
                                  B(0), B(1),
                                  pim))

# Add ImuFactor for interval 2 (t=1 to t=2)
graph.add(gtsam.CombinedImuFactor(X(1), V(1),
                          X(2), V(2),
                          B(1), B(2),
                                  pim))


In [ ]:
# Create initial estimate
initial_estimate = gtsam.Values()
initial_estimate.insert(X(0), pose_0)
initial_estimate.insert(V(0), vel_0)

# Initial estimate for t=1, t=2 (assuming constant velocity)
for k in [1,2]:
  pose_k = gtsam.Pose3(gtsam.Rot3(), gtsam.Point3(0, 0, 0) + vel_0 * dt)
  vel_k = vel_0
  initial_estimate.insert(X(k), pose_k)
  initial_estimate.insert(V(k), vel_k)

# Initial estimate for bias in interval 0, 1 and 2:
initial_estimate.insert(B(0), bias_0)
initial_estimate.insert(B(1), bias_0)
initial_estimate.insert(B(2), bias_0)
print("Initial Estimate:\n", initial_estimate)

In [ ]:
optimizer = gtsam.GaussNewtonOptimizer(graph, initial_estimate) # Force error if indeterminate
result = optimizer.optimize()

# Print results
print("\nOptimized Result:\n", result)

In [ ]:
# Analyze marginals (covariances)
marginals = gtsam.Marginals(graph, result)

# Get covariance for bias in interval 1
bias_cov_1 = marginals.marginalCovariance(gtsam.symbol_shorthand.B(1))
print("\nCovariance for Bias at t=1:\n", np.round(1e9*bias_cov_1))

# Get covariance for bias in interval 2
bias_cov_2 = marginals.marginalCovariance(gtsam.symbol_shorthand.B(2))
print("\nCovariance for Bias at t=2:\n", np.round(1e9*bias_cov_2))

In [ ]:
# The entire posterior over *all* variables is encoded in the Bayes Tree,
# at the end of this print. From it all marginals are computed.
marginals.print()